# Data Processes

The notebook below take in the raw soil moisture and meterological data. It outputs 3 csv files for each station:

1. The raw merged data
2. The merged data that has been cleaned
3. The merged and cleaned data that has had all NaN values simulated by a linear approximation


In [ ]:
# Imports
import os
import datetime
import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [ ]:
# Mounting Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/My Drive/TX_DATA

/content/drive/My Drive/TX_DATA


## Section 1

Here we read both the meterological and soil moisture datasets, convert their values to floats and examine the number of NA's in each column.



In [ ]:
met = {}
for index in range(0, 6):
  df = pd.read_csv('Uncleaned_data/MET_' + str(index + 1) + '.dat', sep = ",", parse_dates=["Date"], index_col="Date")
  met['Station' + str(index + 1)] = df

In [ ]:
sm = {}
for index in range(0, 6):
  df = pd.read_csv('Uncleaned_data/SM_' + str(index + 1) + '.dat', sep = ",", parse_dates=["Date"], index_col="Date")
  sm['Station' + str(index + 1)] = df

In [ ]:
for station, df in met.items():
  print(df.dtypes)

In [ ]:
for station, df in sm.items():
  print(df.dtypes)

In [ ]:
for station, df in met.items():
  for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
for station, df in sm.items():
  for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
for station, df in met.items():
  print(df.dtypes)

In [ ]:
for station, df in sm.items():
  print(df.dtypes)

In [ ]:
for station, df in met.items():
  print(station)
  print(df.isna().sum())

In [ ]:
for station, df in sm.items():
  print(station)
  print(df.isna().sum())

In [ ]:
for station, df in met.items():
  print(station)
  print(len(df))

In [ ]:
for station, df in sm.items():
  print(station)
  print(len(df))

## Section 2

In this section we compute the intersection of all station data frame indexes and store the merged data as csv files.

In [ ]:
# Create union of all indexes
index_union = pd.Index([])

for station, df in met.items():
  index_union = index_union.union(df.index)

for station, df in sm.items():
  index_union = index_union.union(df.index)

In [ ]:
# Create intersection of all indexes
index_int = index_union

for station, df in met.items():
  index_int = index_int.intersection(df.index)

for station, df in sm.items():
  index_int = index_int.intersection(df.index)

In [ ]:
#Set each dfs index to equal the intersection
for key in met.keys():
    met[key] = met[key].loc[index_int]
    sm[key] = sm[key].loc[index_int]

In [ ]:
#Now Merge the met and sm data frames
dfs = {}
for key in met.keys():
  dfs[key] = pd.concat([sm[key], met[key]], axis=1)



In [ ]:
# store raw merged dfs


for key in dfs.keys():
  dfs[key].to_csv( key + '_raw_merged.csv')


## Section 3

In this section we remove the Flag column, forward fill missing values up to two days and drop any indexes in which all features are N/A.

When then save the data as csv files.

In [ ]:
# remove flag
for key in dfs.keys():
  dfs[key].columns = dfs[key].columns.str.replace(' ', '')
  dfs[key].pop('Flag')

In [ ]:

for key in dfs.keys():
  dfs[key] = dfs[key].ffill(limit = 48)

In [ ]:
for key in dfs.keys():
  dfs[key].dropna(inplace = True, how = 'all')

In [ ]:
#add longitude and latitude

position_dict = {"Station1": [30.3989,-98.6105],
                 "Station2": [30.4193,-98.8046],
                 "Station3": [30.4421,-98.8427],
                 "Station4": [30.4600, -98.9407],
                 "Station5": [30.2454,-98.7059],
                 "Station6": [30.2758,-98.7242]
                 }

for key in dfs.keys():
  dfs[key]["Latitude"] = position_dict[key][0]
  dfs[key]["Longitude"] = position_dict[key][1]

'''
for station, df in dfs.items():
    lat = df.pop('Latitude')
    lon = df.pop('Longitude')
    df["x_cord"] = np.cos(lat) * np.cos(lon)
    df["y_cord"] = np.sin(lat) * np.cos(lon)
    df["z_cord"] = np.sin(lon)
'''

'\nfor station, df in dfs.items():\n    lat = df.pop(\'Latitude\')\n    lon = df.pop(\'Longitude\')\n    df["x_cord"] = np.cos(lat) * np.cos(lon)\n    df["y_cord"] = np.sin(lat) * np.cos(lon)\n    df["z_cord"] = np.sin(lon)\n'

In [ ]:
for key in dfs.keys():
  dfs[key].to_csv( key + '_cleaned_merge_data.csv')

## Section4

In this section we identify large numbers of sequential N/A values for all fields, create a linear model for each of these fields, train these models on indexes where all the values are non N/A, and then fill in the N/A values with predictions.

Then we store the dataframes as csv files.

In [ ]:
combined_df = merged_df = pd.concat(dfs.values(), axis=0)
combined_df.reset_index(drop=True, inplace=True)

combined_df

,Ppt,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Ppt,Tair,RH,Windspeed,Winddirection,Srad,Latitude,Longitude
0,0.0,0.139,0.178,0.148,0.152,2.81,4.40,5.77,10.57,0.0,-1.090,81.50,1.052,52.27,0.63,30.3989,-98.6105
1,0.0,0.139,0.178,0.148,0.152,2.86,4.38,5.71,10.51,0.0,-1.038,81.70,0.959,46.71,0.62,30.3989,-98.6105
2,0.0,0.139,0.178,0.148,0.152,2.89,4.35,5.66,10.47,0.0,-0.981,82.00,1.062,52.04,0.60,30.3989,-98.6105
3,0.0,0.139,0.178,0.148,0.152,2.90,4.33,5.62,10.41,0.0,-0.814,81.90,0.887,58.91,0.64,30.3989,-98.6105
4,0.0,0.139,0.178,0.148,0.152,2.96,4.32,5.59,10.36,0.0,-0.805,90.00,0.828,16.55,0.20,30.3989,-98.6105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
348084,0.0,0.104,0.101,0.090,0.062,35.04,34.77,33.60,30.74,0.0,31.410,100.00,0.122,181.60,0.02,30.2758,-98.7242
348085,0.0,0.104,0.100,0.090,0.062,34.52,34.46,33.64,30.80,0.0,30.310,100.00,0.013,184.30,0.00,30.2758,-98.7242
348086,0.0,0.103,0.100,0.090,0.062,33.99,34.07,33.58,30.86,0.0,29.990,2.55,0.309,168.60,0.00,30.2758,-98.7242
348087,0.0,0.103,0.100,0.090,0.062,33.47,33.67,33.45,30.93,0.0,29.450,100.00,1.252,172.70,0.00,30.2758,-98.7242


In [ ]:
combined_df.isna().sum()

Ppt                  0
SWC_5                0
SWC_10               0
SWC_20           10057
SWC_50           58441
T_5                  0
T_10                 0
T_20                 0
T_50             58441
Ppt                  0
Tair                 0
RH                   0
Windspeed            0
Winddirection        0
Srad                 0
Latitude             0
Longitude            0
dtype: int64

In [ ]:
combined_df.columns = combined_df.columns.str.replace(' ', '')

In [ ]:
filtered_df = combined_df[pd.notna(combined_df['SWC_50']) & pd.notna(combined_df['SWC_20']) & pd.notna(combined_df['T_50'])]

In [ ]:

LinModel_SWC50 = LinearRegression(fit_intercept=True)
LinModel_T50 = LinearRegression(fit_intercept=True)
df = filtered_df.copy()
y1 = df.pop("SWC_50")
y2 = df.pop("T_50")
X = df
LinModel_SWC50.fit(X, y1)
LinModel_T50.fit(X, y2)

LinearRegression()

In [ ]:
X_test = combined_df[pd.isna(combined_df['SWC_50']) & pd.isna(combined_df['T_50'])]
X_test.pop("SWC_50")
X_test.pop("T_50")

172766   NaN
172767   NaN
172768   NaN
172769   NaN
172770   NaN
          ..
231202   NaN
231203   NaN
231204   NaN
231205   NaN
231206   NaN
Name: T_50, Length: 58441, dtype: float64

In [ ]:
X_test = dfs["Station4"]
X_test.pop("SWC_50")
X_test.pop("T_50")

2015-01-01 00:00:00   NaN
2015-01-01 01:00:00   NaN
2015-01-01 02:00:00   NaN
2015-01-01 03:00:00   NaN
2015-01-01 04:00:00   NaN
                       ..
2021-08-31 20:00:00   NaN
2021-08-31 21:00:00   NaN
2021-08-31 22:00:00   NaN
2021-08-31 23:00:00   NaN
2021-09-01 00:00:00   NaN
Name: T_50, Length: 58441, dtype: float64

In [ ]:
y1_pred = LinModel_SWC50.predict(X_test)
y2_pred = LinModel_T50.predict(X_test)

In [ ]:
(y1_pred < 0).sum()

4

In [ ]:
(y2_pred < 0).sum()

0

In [ ]:
y1_pred[y1_pred < 0] = 0

In [ ]:
LinModel_SWC20 = LinearRegression(fit_intercept=True)
df = filtered_df.copy()
y3 = df.pop("SWC_20")
X = df
LinModel_SWC20.fit(X, y3)


LinearRegression()

In [ ]:
X_test = dfs["Station5"][pd.isna(dfs["Station5"]['SWC_20'])].copy()
X_test.pop("SWC_20")
y3_pred = LinModel_SWC20.predict(X_test)

In [ ]:
X_test = dfs["Station1"][pd.isna(dfs["Station1"]['SWC_20'])].copy()
X_test.pop("SWC_20")
y4_pred = LinModel_SWC20.predict(X_test)

In [ ]:
(y3_pred < 0).sum()

0

In [ ]:
(y4_pred < 0).sum()

0

In [ ]:
dfs["Station4"]["SWC_50"] = y1_pred
dfs["Station4"]["T_50"] = y2_pred

In [ ]:
null_index = dfs["Station5"]['SWC_20'].isna()
dfs["Station5"].loc[null_index, 'SWC_20'] = y3_pred

In [ ]:
null_index = dfs["Station1"]['SWC_20'].isna()
dfs["Station1"].loc[null_index, 'SWC_20'] = y4_pred

In [ ]:
for station, df in dfs.items():
  print(station)
  print(df.isna().sum())

In [ ]:
for key in dfs.keys():
  dfs[key].to_csv( key + '_simulated_cleaned_merged_data.csv')

In [ ]:
for station, df in dfs.items():
  print(station)
  print(len(df))